# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Optional: display available keys and summary
print(f"Dataset published on: {metadata.datePublished}")
print(f"Fields (as per Croissant schema): {metadata.personalSensitiveInformation}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, datasets can contain one or more record sets (tables), each identified by a unique `@id`. Let's explore the available record sets and their fields using their `@id` references.

In [ ]:
# List all available record sets
record_sets = dataset.record_sets()
print("Record Sets:")
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', '[no name]')}, description: {rs.get('description', '[no description]')}")

# For each record set, list fields and columns by @id
for record_set in record_sets:
    print(f"\nRecordSet @id: {record_set['@id']}")
    fields = record_set.get('field', [])
    columns = record_set.get('column', [])
    if fields:
        print("Fields:")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"  - Field @id: {field_id}")
    else:
        print("Fields: [none listed]")
    if columns:
        print("Columns:")
        for column in columns:
            column_id = column['@id'] if isinstance(column, dict) else column
            print(f"  - Column @id: {column_id}")
    else:
        print("Columns: [none listed]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview above.

For demonstration, we'll extract data from the main survey record set (identified by its `@id`). Make sure to update the ID to match what you found above.

In [ ]:
# List all record sets again to select one for extraction
record_sets = dataset.record_sets()
record_set_ids = [rs['@id'] for rs in record_sets]
print("RecordSet IDs:", record_set_ids)

# Choose the main record set (update this if needed based on data overview)
main_record_set_id = record_set_ids[0] if record_set_ids else None
# Optionally, extract all tables
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"{record_set_id} columns: {df.columns.tolist()}")
if main_record_set_id:
    print("Top 5 rows:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps, such as filtering records by specific criteria, normalizing numeric fields, and grouping records.

We'll demonstrate by filtering high-scoring GAD-7 entries (general anxiety disorder score), normalizing the scores, and grouping by village.

In [ ]:
# EDA Example: Filter, normalize, group
# --- Replace with actual field @id based on overview if needed ---
record_set_id = main_record_set_id
df = dataframes[record_set_id].copy()

# Example numeric field: the GAD-7 score (use correct @id)
numeric_field_id = 'GAD7_score'  # Replace with actual @id if needed
if numeric_field_id not in df.columns:
    numeric_field_id = df.columns[-1]  # Fallback: last column
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

# Group by village (update with actual field @id)
group_field_id = 'village'  # Replace with actual @id if needed
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions to gain further insight.
Here, we'll plot the distribution of GAD-7 scores and their relationship to PHQ-9 scores (depression). Update the field @ids as necessary.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of GAD-7 scores
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Plot relationship between GAD-7 and PHQ-9 scores
phq9_field_id = 'PHQ9_score'  # Replace with actual @id if needed
if phq9_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df[numeric_field_id], y=df[phq9_field_id])
    plt.title(f"{numeric_field_id} vs {phq9_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel(phq9_field_id)
    plt.show()
else:
    print(f"Field {phq9_field_id} not found in DataFrame columns. Available columns: {df.columns.tolist()}")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The FAIR² Kilifi dataset provides rich survey data on mental health, with fields for demographics and psychological indicators such as GAD-7, PHQ-9, and PSQ scores.
* Using Croissant `@id` references, all entities can be accessed programmatically and reproducibly.
* Exploratory analysis shows how to filter, normalize, and visualize scores; useful trends can be grouped by personal and community-level fields.
* The resulting insights can inform public health and research initiatives concerning mental health in Kilifi County, Kenya.

&nbsp;
